## 1 — Imports

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from pathlib import Path
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import TextLoader
import glob
from openai import OpenAI
import os
from embeddings.nvidia_embeddings import NVIDIAEmbeddings

from pinecone import ServerlessSpec
import pinecone
from pinecone import Pinecone

from embeddings.dense_embed import DenseEmbeddings
from sparse_index import SparseEmbeddings


## 2 — Config

In [ ]:
NVIDIA_API_KEY=os.environ.get("NVIDIA_API_KEY")
PINECONE_API_KEY =os.environ.get("PINECONE_API_KEY")

In [ ]:
index_name=os.environ.get("PINECONE_INDEX_NAME")

In [30]:
print(index_name)

sift


In [31]:
pc = Pinecone(api_key=PINECONE_API_KEY)

# Creating an index

In [32]:
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",          # required
        dimension=2048,               # ← change to your actual dense dim
        metric="dotproduct",          # ← mandatory for hybrid
        spec=ServerlessSpec(
            cloud="aws",              # or "gcp" / "azure"
            region="us-east-1"        # choose a region close to you
        )
    )

## 3 — Load documents

In [33]:
output_path = "../../data/parsed"
pages_path = glob.glob(f"{output_path}/*.md")

## 4 — Split

In [34]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = []

for page in pages_path:
    text_loader = TextLoader(
        page,
        encoding="utf-8"
    )

    docs.extend(text_loader.load())

chunks = splitter.split_documents(docs)

In [35]:
texts = [chunk.page_content for chunk in chunks]

## 5 — Embeddings

In [36]:
dense = DenseEmbeddings(nvidia_api_key=NVIDIA_API_KEY)
sparse = SparseEmbeddings(pc_api_key=PINECONE_API_KEY)

In [37]:
dense_embeddings = dense.generate_embeddings(texts)
sparse_embeddings = sparse.generate_embeddings(texts)

In [38]:
records = []

for i, chunk in enumerate(chunks):

    records.append({
        "id": f"hrbird-{i}",
        "values": dense_embeddings[i],
        "sparse_values": sparse_embeddings[i],
        "metadata": {
            "text": chunk.page_content,
            **chunk.metadata
        }
    })

In [ ]:
from pinecone import Pinecone

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index = pc.Index(index_name)

In [40]:
pc.describe_index(index_name)

{
    "name": "sift",
    "metric": "dotproduct",
    "host": "sift-pyq2x42.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 2048,
    "deletion_protection": "disabled",
    "tags": null
}

In [41]:
index.upsert(
    vectors=records,
    namespace="hrbird"
)

{'upserted_count': 53}

In [42]:
print(len(chunks))
print(len(dense_embeddings))
print(len(sparse_embeddings))
print(len(dense_embeddings[0]))
print(sparse_embeddings[0])

53
53
53
2048
{'indices': [38699465, 108818272, 192819851, 372196114, 377896724, 512480045, 521357180, 577606826, 814527388, 1236476080, 1260363956, 1299665196, 1491351846, 1608361540, 1734098574, 1746685302, 1870141158, 1935398516, 2021799277, 2031222688, 2076516198, 2079815196, 2142429621, 2274901607, 2451242591, 2523205057, 2701069243, 2875561829, 3024548359, 3046140599, 3160171665, 3327252652, 3392840550, 3525523449, 3606427239, 3611963450, 3619727943, 4038825476, 4259959689, 4279915734], 'values': [3.875, 5.3515625, 1.6572266, 4.2851562, 4.3007812, 3.9199219, 3.4472656, 3.5917969, 0.51220703, 2.7382812, 3.1679688, 0.22497559, 1.6552734, 4.1601562, 4.7851562, 1.6630859, 1.2216797, 1.7675781, 0.38378906, 2.9863281, 5.4101562, 3.1171875, 3.1621094, 3.1835938, 3.4394531, 3.2519531, 2.3359375, 3.9277344, 1.5234375, 1.3867188, 4.2695312, 0.80566406, 3.4921875, 2.4101562, 4.9179688, 2.5859375, 3.2109375, 2.0859375, 3.3789062, 1.0712891]}


In [ ]:
pc.similarity_search_woi